# PyDI Data Integration Workflow: Music

This notebook demonstrates how PyDI is used for end-to-end data integration. We'll work with music datasets to showcase the data integration pipeline from schema and entity matching to data fusion.

## Table of Contents
  - [Datasets](#datasets)
- [Part 1: Schema Matching and Value Normalization](#part-1-schema-matching-and-value-normalization)
  - [Step 1: Load Target Schema and Normalization Spec](#step-1-load-target-schema-and-normalization-spec)
  - [Step 2: Load Source Datasets](#step-2-load-source-datasets)
  - [Step 3: LLM-Based Schema Matching](#step-3-llm-based-schema-matching)
  - [Step 4: Schema Matching Evaluation](#step-4-evaluate-schema-matching-against-gold-mapping)
  - [Step 5: Translate and Normalize](#step-5-translate-and-normalize)
- [Part 2: Data Loading and Profiling](#part-2-data-loading-and-profiling)
- [Part 3: Entity Matching](#part-3-entity-matching)
  - [Step 1: Blocking](#step-1-blocking)
  - [Step 2: Blocking Evaluation](#step-2-evaluate-blocking-against-ground-truth)
  - [Step 3: Entity Matching with Comparators](#step-3-entity-matching-with-comparators)
  - [Step 4: Entity Matching Evaluation](#step-4-evaluate-matching-against-ground-truth)
- [Part 4: Data Fusion](#part-4-data-fusion)
  - [Step 1: Define Fusion Strategy](#step-1-define-fusion-strategy)
  - [Step 2: Run Fusion](#step-2-run-fusion)
  - [Step 3: Data Fusion Evaluation](#step-3-evaluate-data-fusion)

## Part 1: Schema Matching and Value Normalization

In [1]:
from pathlib import Path
import time
start_time = time.time()
# Paths relative to this notebook
NOTEBOOK_DIR = Path(".").resolve()
INPUT_DIR = NOTEBOOK_DIR / "input"
OUTPUT_DIR = NOTEBOOK_DIR / "output"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

In [2]:
import pandas as pd
import json
from PyDI.schemamatching import LLMBasedSchemaMatcher, SchemaMappingEvaluator, SchemaTranslator
from PyDI.normalization import load_normalization_spec
from langchain_openai import ChatOpenAI

/Users/aaronsteiner/Documents/GitHub/PyDI/env/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Step 1: Load Target Schema and Normalization Spec

In [3]:
# Load the JSON Schema (used for both matching and normalization)
with open(INPUT_DIR / "schemamatching" / "target_schema.json") as f:
    target_schema = json.load(f)

# Load NormalizationSpec from the same schema
spec = load_normalization_spec(INPUT_DIR / "schemamatching" / "target_schema.json")

spec.set_column("tracks", output_type="list")
spec.set_column("release-date", output_type="string")

target_columns = list(spec.columns.keys())

# Create empty target DataFrame for schema matching
df_target = pd.DataFrame(columns=target_columns)
df_target.attrs["dataset_name"] = "target_schema"


## Step 2: Load Source Datasets

In [4]:
from PyDI.io import load_xml, load_csv, load_json
mbrainz = load_csv(INPUT_DIR / "data" / "musicbrainz.csv")
mbrainz.attrs["dataset_name"] = "musicbrainz"
mbrainz.head()

,Attribute_1,Attribute_2,Attribute_3,Attribute_4,Attribute_5,Attribute_6,Attribute_9
0,mbrainz_1,Fermats Theorem / Sight Beyond,John B,1996-01-01,United Kingdom of Great Britain and Northern I...,1055,"['Fermats Theorem', 'Sight Beyond']"
1,mbrainz_2,Tempest / Inner Sense,Psychosis,1998-12-14,United Kingdom of Great Britain and Northern I...,724,"['Tempest', 'Inner Sense']"
2,mbrainz_3,The Sign's Alive,Lypid,2000-09-05,United States of America,2384,"[""The Sign's Alive (original mix)"", ""The Sign'..."
3,mbrainz_4,Surrender,Petalpusher,1999-04-27,United States of America,1626,"['Surrender (Petalpusher original)', ""Surrende..."
4,mbrainz_6,Unreasonable Behaviour,"Garnier, Laurent",2000-07-24,France,4145,"['The Warning', 'City Sphere', 'Forgotten Thou..."


In [5]:
import re
# Clean column names from XML namespaces
def strip_all_ns(col):
    """
    Remove ALL {namespace} prefixes from a column name string.
    Example:
      '{ns}medium-list_{ns}medium_{ns}track' -> 'medium-list_medium_track'
    """
    if not isinstance(col, str):
        return col
    # remove every occurrence of {...}
    return re.sub(r"\{[^}]+\}", "", col)

mbrainz = mbrainz.rename(columns=strip_all_ns)
mbrainz.head()

,Attribute_1,Attribute_2,Attribute_3,Attribute_4,Attribute_5,Attribute_6,Attribute_9
0,mbrainz_1,Fermats Theorem / Sight Beyond,John B,1996-01-01,United Kingdom of Great Britain and Northern I...,1055,"['Fermats Theorem', 'Sight Beyond']"
1,mbrainz_2,Tempest / Inner Sense,Psychosis,1998-12-14,United Kingdom of Great Britain and Northern I...,724,"['Tempest', 'Inner Sense']"
2,mbrainz_3,The Sign's Alive,Lypid,2000-09-05,United States of America,2384,"[""The Sign's Alive (original mix)"", ""The Sign'..."
3,mbrainz_4,Surrender,Petalpusher,1999-04-27,United States of America,1626,"['Surrender (Petalpusher original)', ""Surrende..."
4,mbrainz_6,Unreasonable Behaviour,"Garnier, Laurent",2000-07-24,France,4145,"['The Warning', 'City Sphere', 'Forgotten Thou..."


In [6]:
lastfm = load_csv(INPUT_DIR / "data" / "lastfm.csv")
lastfm.attrs["dataset_name"] = "lastfm"
lastfm.head()

,item_code,album_title,band,album_length,tracks_track-name
0,lastFM_1,John B - Fermats Theorem / Sight Beyond,John B,903.0,"['Fermats Theorem', 'Sight Beyond']"
1,lastFM_2,Tempest / Inner Sense,Psychosis,734.0,"['Tempest', 'Inner Sense']"
2,lastFM_4,Petalpusher - Surrender,Petalpusher,1626.0,"['Surrender (Petalpusher Original)', ""Surrende..."
3,lastFM_8,in the spirit,R. Trent,1265.0,"['In The Spirit (The Full Experience)', 'In Th..."
4,lastFM_11,Come Of Age,S. Vitus Dance,1378.0,"['Bliss', 'Tunnel Vision', 'Catch The Sun', 'M..."


In [7]:
discogs = load_csv(INPUT_DIR / "data" / "discogs.csv")
discogs.attrs["dataset_name"] = "discogs"
discogs.head()

,rec_uid,title_str,performer,pub_dt,origin_loc,duration,imprint,category,tracks_track-name
0,discogs_3,Fermats Theorem / Sight Beyond,John B,1996-01-01,UK,0,New Identity Recordings,Electronic,"['Fermats Theorem', 'Sight Beyond']"
1,discogs_4,Tempest / Inner Sense,Psychosis,1998-01-01,UK,0,Renegade Hardware,Electronic,"['Tempest', 'Inner Sense']"
2,discogs_5,The Sign's Alive,Lypid,2000-09-05,United States of America,0,Statra Recordings,Electronic,"[""The Sign's Alive (Original Mix)"", ""The Sign'..."
3,discogs_6,Surrender,Petalpusher,1999-04-27,United States of America,1626,Naked Music Recordings,Electronic,"['Surrender (Petalpusher Original)', ""Surrende..."
4,discogs_11,Unreasonable Behaviour,Laurent Garnier,2000-06-01,France,5938,F Communications,Electronic,"['The Warning', 'City Sphere', 'Forgotten Thou..."


## Step 3: LLM-Based Schema Matching

In [8]:
from dotenv import load_dotenv
load_dotenv()

# Initialize matcher with target schema for better context
matcher = LLMBasedSchemaMatcher(
    chat_model=ChatOpenAI(model="gpt-5.5"),
    num_rows=40,
    target_schema=target_schema,
)

# Match mbrainz dataset
mbrainz_mapping = matcher.match(mbrainz, df_target)

mbrainz_mapping

,source_dataset,source_column,target_dataset,target_column,score,notes
0,musicbrainz,Attribute_1,target_schema,id,0.95,llm_based_matching
1,musicbrainz,Attribute_2,target_schema,name,0.95,llm_based_matching
2,musicbrainz,Attribute_3,target_schema,artist,0.95,llm_based_matching
3,musicbrainz,Attribute_4,target_schema,release-date,0.95,llm_based_matching
4,musicbrainz,Attribute_5,target_schema,release-country,0.95,llm_based_matching
5,musicbrainz,Attribute_6,target_schema,duration,0.95,llm_based_matching
6,musicbrainz,Attribute_9,target_schema,tracks,0.95,llm_based_matching


In [9]:
lastfm_mapping = matcher.match(lastfm, df_target)
lastfm_mapping

,source_dataset,source_column,target_dataset,target_column,score,notes
0,lastfm,item_code,target_schema,id,0.95,llm_based_matching
1,lastfm,album_title,target_schema,name,0.95,llm_based_matching
2,lastfm,band,target_schema,artist,0.95,llm_based_matching
3,lastfm,album_length,target_schema,duration,0.95,llm_based_matching
4,lastfm,tracks_track-name,target_schema,tracks,0.95,llm_based_matching


In [10]:
discogs_mapping = matcher.match(discogs, df_target)
discogs_mapping

,source_dataset,source_column,target_dataset,target_column,score,notes
0,discogs,rec_uid,target_schema,id,0.95,llm_based_matching
1,discogs,title_str,target_schema,name,0.95,llm_based_matching
2,discogs,performer,target_schema,artist,0.95,llm_based_matching
3,discogs,pub_dt,target_schema,release-date,0.95,llm_based_matching
4,discogs,origin_loc,target_schema,release-country,0.95,llm_based_matching
5,discogs,duration,target_schema,duration,0.95,llm_based_matching
6,discogs,imprint,target_schema,label,0.95,llm_based_matching
7,discogs,category,target_schema,genre,0.95,llm_based_matching
8,discogs,tracks_track-name,target_schema,tracks,0.95,llm_based_matching


## Step 4: Evaluate Schema Matching Against Gold Mapping


In [11]:
# Load the manually curated schema-matching gold standard
with open(INPUT_DIR / "schemamatching" / "sm_mapping_gold.json") as f:
    schema_mapping_gold = json.load(f)

schema_mapping_gold_df = pd.DataFrame(schema_mapping_gold["mappings"])
schema_mapping_predictions = pd.concat(
    [mbrainz_mapping, lastfm_mapping, discogs_mapping],
    ignore_index=True,
)

schema_matching_metrics = SchemaMappingEvaluator.evaluate(
    schema_mapping_predictions,
    schema_mapping_gold_df,
    complete=True,
)

schema_matching_summary = pd.DataFrame([
    {
        **schema_matching_metrics,
        "predicted_mappings": len(schema_mapping_predictions),
        "gold_mappings": len(schema_mapping_gold_df),
    }
])

per_source_schema_matching = pd.DataFrame([
    {
        "source_dataset": source_dataset,
        **SchemaMappingEvaluator.evaluate(
            schema_mapping_predictions[
                schema_mapping_predictions["source_dataset"] == source_dataset
            ],
            schema_mapping_gold_df[
                schema_mapping_gold_df["source_dataset"] == source_dataset
            ],
            complete=True,
        ),
    }
    for source_dataset in sorted(schema_mapping_gold_df["source_dataset"].unique())
])

display(schema_matching_summary)
per_source_schema_matching


,precision,recall,f1,correct,matched,correct_total,missing,predicted_mappings,gold_mappings
0,1.0,1.0,1.0,21,21,21,0,21,21


,source_dataset,precision,recall,f1,correct,matched,correct_total,missing
0,discogs,1.0,1.0,1.0,9,9,9,0
1,lastfm,1.0,1.0,1.0,5,5,5,0
2,musicbrainz,1.0,1.0,1.0,7,7,7,0


## Step 5: Translate and Normalize


In [12]:
translator = SchemaTranslator()

# Translate + normalize each dataset with its own mapping

spec.set_column("release-country", country_format="name")
spec.set_column("release-date", output_type="string")

discogs_normalized = translator.translate(
    discogs,
    discogs_mapping,
    normalize=spec,
    on_failure="keep"
)

lastfm_normalized = translator.translate(
    lastfm, lastfm_mapping,
    normalize=spec, on_failure="keep"
)

mbrainz_normalized = translator.translate(
    mbrainz, mbrainz_mapping,
    normalize=spec, on_failure="keep"
)

In [13]:
discogs_cols = [c for c in target_columns if c in discogs_normalized.columns]
lastfm_cols = [c for c in target_columns if c in lastfm_normalized.columns]
mbrainz_cols = [c for c in target_columns if c in mbrainz_normalized.columns]

In [14]:
# Only keep target columns
mbrainz = mbrainz_normalized[mbrainz_cols].copy()
lastfm = lastfm_normalized[lastfm_cols].copy()
discogs = discogs_normalized[discogs_cols].copy()

import ast

def parse_track_list(value):
    if isinstance(value, list):
        items = value
    elif pd.isna(value):
        return []
    elif isinstance(value, str):
        text = value.strip()
        if not text:
            return []
        try:
            parsed = ast.literal_eval(text)
            items = parsed if isinstance(parsed, list) else [parsed]
        except (SyntaxError, ValueError):
            items = [part.strip() for part in text.split("|")]
    else:
        items = [value]

    cleaned = []
    seen = set()
    for item in items:
        if item is None or pd.isna(item):
            continue
        title = str(item).strip()
        if not title:
            continue
        key = re.sub(r"\s+", " ", title.casefold())
        if key in seen:
            continue
        seen.add(key)
        cleaned.append(title)
    return cleaned

for dataset in (mbrainz, discogs, lastfm):
    if "tracks" in dataset.columns:
        dataset["tracks"] = dataset["tracks"].apply(parse_track_list)


In [15]:
# Keep release dates in the target schema format (YYYY-MM-DD) before translation.
def normalize_iso_date_column(df, column):
    if column not in df.columns:
        return df

    original = df[column]
    text = original.astype("string").str.strip().str.replace(r"-00", "-01", regex=True)
    text = text.str.replace(r"^([0-9]{4})$", r"\1-01-01", regex=True)
    text = text.str.replace(r"^([0-9]{4}-[0-9]{2})$", r"\1-01", regex=True)
    missing = original.isna() | text.eq("")
    parsed = pd.to_datetime(text, errors="coerce", format="%Y-%m-%d")
    normalized = parsed.dt.strftime("%Y-%m-%d")
    df[column] = normalized.where(parsed.notna(), original)
    df.loc[missing, column] = pd.NA
    return df


for dataset in (mbrainz, discogs, lastfm):
    normalize_iso_date_column(dataset, "release-date")

discogs.iloc[0]["release-date"]

'1996-01-01'

## Part 2: Data Loading and Profiling

In [16]:
# Display basic information
datasets = [discogs, mbrainz, lastfm]
names = ["Discogs", "MusicBrainz", "Last.fm"]

total_records = sum(len(df) for df in datasets)
print(f"Total records across all datasets: {total_records:,}")

Total records across all datasets: 37,255


In [17]:
from PyDI.utils import DataProfiler

# Initialize the DataProfiler
profiler = DataProfiler()

for df, name in zip(datasets, names):
    profile = profiler.summary(df) # automatically prints some statistics and returns object containing stats

display(profile)

discogs:
  Rows: 22,627
  Columns: 9
  Total nulls: 2,834
  Null percentage: 1.4%
  Null counts per column:
    release-date: 2,234 (9.9%)
    release-country: 600 (2.7%)

musicbrainz:
  Rows: 4,763
  Columns: 7
  Total nulls: 1,061
  Null percentage: 3.2%
  Null counts per column:
    release-date: 313 (6.6%)
    release-country: 748 (15.7%)

lastfm:
  Rows: 9,865
  Columns: 5
  Total nulls: 5,230
  Null percentage: 10.6%
  Null counts per column:
    duration: 5,230 (53.0%)



{'rows': 9865,
 'columns': 5,
 'nulls_total': 5230,
 'nulls_per_column': {'id': 0,
  'name': 0,
  'artist': 0,
  'duration': 5230,
  'tracks': 0},
 'dtypes': {'id': 'object',
  'name': 'object',
  'artist': 'object',
  'duration': 'float64',
  'tracks': 'object'}}

## Part 3: Entity Matching

### Step 1: Blocking

In [18]:
# Set up logging
import logging

import os
os.makedirs('output/logs', exist_ok=True)

logging.basicConfig(
    level=logging.WARNING, # Alternatively, use logging.DEBUG for more verbosity
    format='[%(levelname)-5s] %(name)s - %(message)s',
    handlers=[
          logging.FileHandler('output/logs/pydi.log'),  # Save to file
          logging.StreamHandler()                      # Display on console
      ],
    force=True
)

In [19]:
# Import blocking methods
from PyDI.entitymatching import StandardBlocker, SortedNeighbourhoodBlocker, TokenBlocker, EmbeddingBlocker
import re

# Standard Blocking - Longest Token in Name
# Add name_longest_token directly to the original dataframes
def get_longest_token(name):
    tokens = re.split(r"[^A-Za-z0-9_']+", str(name))    
    tokens = [t for t in tokens if t]
    return max(tokens, key=len) if tokens else ''

mbrainz['name_longest_token'] = mbrainz['name'].apply(get_longest_token)
discogs['name_longest_token'] = discogs['name'].apply(get_longest_token)
lastfm['name_longest_token'] = lastfm['name'].apply(get_longest_token)

standard_blocker_m2d = StandardBlocker(
    mbrainz, discogs,
    on=['name_longest_token'],
    batch_size=1000,
    output_dir=OUTPUT_DIR / "blocking-evaluation",
    id_column='id'
 )

standard_blocker_m2l = StandardBlocker(
    mbrainz, lastfm,
    on=['name_longest_token'],
    batch_size=1000,
    output_dir=OUTPUT_DIR / "blocking-evaluation",
    id_column='id'
 )

### Step 2: Evaluate Blocking Against Ground Truth

In [20]:
import pandas as pd
from PyDI.io import load_csv
from PyDI.entitymatching import EntityMatchingEvaluator

# Evaluate the blocker against the held-out test pairs for each dataset pair
pairs = {
    "m2d": (standard_blocker_m2d, "musicbrainz_2_discogs_val.csv"),
    "m2l": (standard_blocker_m2l, "musicbrainz_2_lastfm_val.csv"),
}

blocking_results = {}
for pair_name, (blocker, gt_file) in pairs.items():
    test_gt = load_csv(
        INPUT_DIR / "entitymatching" / gt_file,
        name=f"test_{pair_name}", header=None, names=['id1', 'id2', 'label'], add_index=False,
    )
    blocking_results[pair_name] = EntityMatchingEvaluator.evaluate_blocking_batched(
        blocker=blocker,
        test_pairs=test_gt,
        out_dir=OUTPUT_DIR / "blocking-evaluation" / pair_name,
    )

blocking_summary = pd.DataFrame(
    [{"dataset_pair": name, **res} for name, res in blocking_results.items()]
)[["dataset_pair", "pair_completeness", "reduction_ratio"]]

print("Summary of Blocking Results:")
display(blocking_summary)

Summary of Blocking Results:


,dataset_pair,pair_completeness,reduction_ratio
0,m2d,1.000000,0.996701
1,m2l,0.929268,0.996718


### Step 3: Entity Matching with Comparators

In [21]:
from PyDI.entitymatching import StringComparator, DateComparator, NumericComparator

# ignore case and punctuation
def normalize_text(s: str) -> str: 
    if s is None:
        return ""
    return re.sub(r"[^\w\s]|_", "", s).lower()

comparators = [
    # Release name — Jaccard
    StringComparator(
        column='name', 
        similarity_function='jaccard',
        preprocess=normalize_text
    ),
    # Release artist — Jaccard
    StringComparator(
        column='artist',
        similarity_function='jaccard',
        preprocess=normalize_text
    ),
    # Release duration — within 10% --> allow 10% deviation
    NumericComparator(
        column='duration',
        method='relative_difference',
        max_difference=0.10
    ),
    # # Track list — overlap
    StringComparator(
        column='tracks',
        similarity_function='jaccard',
        preprocess=normalize_text,
        list_strategy="set_overlap"
    ),
    # Release date — within 2 years
    DateComparator(
        column='release-date',
        max_days_difference=365 * 2
    ),
    # Release country — Jaccard
    StringComparator(
        column='release-country',
        similarity_function='jaccard',
        preprocess=normalize_text
    ),
]

In [22]:
import numpy as np

# Convert lists in mbrainz["duration"] to single integer values (sum if list, else int)

def sum_duration(val):
    if isinstance(val, list):
        return int(np.nansum([int(x) for x in val if str(x).isdigit()]))
    try:
        return int(val)
    except Exception:
        return np.nan

mbrainz["duration"] = mbrainz["duration"].apply(sum_duration)

In [23]:
from PyDI.entitymatching import RuleBasedMatcher

# Run rule-based matching on each candidate pool. lastfm lacks release-date and release-country,
# so it uses a reduced comparator list.
matcher = RuleBasedMatcher()
matching_pairs = {
    "m2d": (mbrainz, discogs, standard_blocker_m2d, comparators, 0.5),
    "m2l": (mbrainz, lastfm, standard_blocker_m2l, comparators[:-2], 0.3),
}

correspondences = {}
for pair_name, (df_left, df_right, blocker, comps, threshold) in matching_pairs.items():
    correspondences[pair_name] = matcher.match(
        df_left=df_left,
        df_right=df_right,
        candidates=blocker,
        comparators=comps,
        weights=None,
        threshold=threshold,
        id_column='id',
    )

correspondences_m2d = correspondences["m2d"]
correspondences_m2l = correspondences["m2l"]

### Step 4: Evaluate Matching Against Ground Truth

In [24]:
from PyDI.entitymatching import MaximumBipartiteMatching

# Evaluate raw and 1:1-refined matching for each dataset pair
clusterer = MaximumBipartiteMatching()
gt_files = {
    "m2d": "musicbrainz_2_discogs_test.csv",
    "m2l": "musicbrainz_2_lastfm_test.csv",
}

matching_rows = []
correspondences_refined = {}
for pair_name, gt_file in gt_files.items():
    gt_test = load_csv(
        INPUT_DIR / "entitymatching" / gt_file,
        name=f"test_{pair_name}", header=None, names=['id1', 'id2', 'label'], add_index=False,
    )
    raw = correspondences[pair_name]
    refined = clusterer.cluster(raw)
    correspondences_refined[pair_name] = refined

    raw_eval = EntityMatchingEvaluator.evaluate_matching(
        correspondences=raw, test_pairs=gt_test,
        out_dir=OUTPUT_DIR / "debug_results_entity_matching" / f"{pair_name}_raw",
    )
    refined_eval = EntityMatchingEvaluator.evaluate_matching(
        correspondences=refined, test_pairs=gt_test,
        out_dir=OUTPUT_DIR / "debug_results_entity_matching" / f"{pair_name}_mbm",
    )
    matching_rows.append({"dataset_pair": pair_name, "stage": "raw", **raw_eval})
    matching_rows.append({"dataset_pair": pair_name, "stage": "mbm_refined", **refined_eval})

correspondences_m2d_post = correspondences_refined["m2d"]
correspondences_m2l_post = correspondences_refined["m2l"]

matching_summary = pd.DataFrame(matching_rows)[["dataset_pair", "stage", "precision", "recall", "f1", "accuracy"]]
print("Matching results (raw vs MBM-refined):")
display(matching_summary)

Matching results (raw vs MBM-refined):


,dataset_pair,stage,precision,recall,f1,accuracy
0,m2d,raw,0.956268,0.984985,0.970414,0.980
1,m2d,mbm_refined,0.979239,0.849850,0.909968,0.944
2,m2l,raw,0.959119,0.915916,0.937020,0.959
3,m2l,mbm_refined,0.989130,0.819820,0.896552,0.937


## Part 4: Data Fusion

In [25]:
mbrainz["mbrainz_id"] = mbrainz["id"]

# Assign trust scores to datasets
mbrainz.attrs["trust_score"] = 3
discogs.attrs["trust_score"] = 1
lastfm.attrs["trust_score"] = 2

all_correspondences = pd.concat([correspondences_m2d, correspondences_m2l], ignore_index=True)
print(f'Total correspondences: {len(all_correspondences):,}')

Total correspondences: 8,475


## Step 1: Define Fusion Strategy 

In [26]:
from PyDI.fusion import DataFusionStrategy, longest_string, shortest_string, prefer_higher_trust, maximum

strategy = DataFusionStrategy('music_fusion_strategy')

def prefer_track_list_by_source(values, *, sources=None, source_datasets=None, **kwargs):
    """Pick one coherent track list instead of unioning near-duplicate track titles."""
    source_priority = ["musicbrainz", "discogs", "lastfm"]
    candidates = []
    for value, record_id in zip(values, sources or []):
        tracks = parse_track_list(value)
        if not tracks:
            continue
        dataset = (source_datasets or {}).get(record_id, "unknown")
        priority = source_priority.index(dataset) if dataset in source_priority else len(source_priority)
        candidates.append((priority, -len(tracks), dataset, record_id, tracks))

    if not candidates:
        return None, 0.0, {"rule": "prefer_track_list_by_source", "reason": "no_valid_track_lists"}

    priority, _, dataset, record_id, tracks = sorted(candidates)[0]
    confidence = 1.0 if priority == 0 else max(0.5, 1.0 - 0.2 * priority)
    return tracks, confidence, {
        "rule": "prefer_track_list_by_source",
        "selected_dataset": dataset,
        "selected_record_id": record_id,
        "num_tracks": len(tracks),
    }

strategy.add_attribute_fuser('name', shortest_string)
strategy.add_attribute_fuser('artist', longest_string)
strategy.add_attribute_fuser('release-date', prefer_higher_trust)
strategy.add_attribute_fuser('release-country', prefer_higher_trust)
strategy.add_attribute_fuser('duration', maximum)
strategy.add_attribute_fuser('tracks', prefer_track_list_by_source)
strategy.add_attribute_fuser('label', longest_string)


## Step 2: Run Fusion

In [27]:
from PyDI.fusion import DataFusionEngine

engine = DataFusionEngine(strategy, debug=True, debug_format='json',debug_file=OUTPUT_DIR / "data_fusion" / "debug_fusion.jsonl")

fused = engine.run(
    datasets=[mbrainz, discogs, lastfm],
    correspondences=all_correspondences,
    id_column="id",
    include_singletons=False,
)

def musicbrainz_cluster_id(sources):
    if not isinstance(sources, (list, tuple, set)):
        return None
    for source_id in sources:
        source_id = str(source_id)
        if source_id.startswith("mbrainz_"):
            return source_id
    return None

# Align fused records with the gold fusion sets, which use the MusicBrainz id.
# Example: [mbrainz_1, discogs_3, lastFM_1] should evaluate as id == mbrainz_1.
fused["id"] = fused["_fusion_sources"].apply(musicbrainz_cluster_id)
fused = fused.dropna(subset=["id"]).copy()

print(f'Fused rows: {len(fused):,}')
display(fused.head(5))


Fused rows: 3,624


,_id,_fusion_sources,_fusion_source_datasets,artist,duration,genre,id,label,mbrainz_id,name,name_longest_token,release-country,release-date,tracks,_fusion_confidence,_fusion_metadata
0,discogs_3,"[mbrainz_1, discogs_3, lastFM_1]","[musicbrainz, discogs, lastfm]",John B,1055.0,Electronic,mbrainz_1,New Identity Recordings,mbrainz_1,Fermats Theorem / Sight Beyond,Fermats,United Kingdom,1996-01-01,"[Fermats Theorem, Sight Beyond]",0.700000,"{'_id_rule': 'first_non_null', '_id_inputs': [..."
1,discogs_111970,"[mbrainz_2, mbrainz_455, mbrainz_19993, mbrain...","[musicbrainz, musicbrainz, musicbrainz, musicb...",Tech Level 2,2510.0,Rock,mbrainz_2,Renegade Hardware,mbrainz_19993,Tempest,Tempest,United Kingdom,1998-12-14,"[Coma Burn, Engravings, Tempest]",0.505882,"{'_id_rule': 'first_non_null', '_id_inputs': [..."
2,discogs_5,"[mbrainz_3, discogs_5]","[musicbrainz, discogs]",Lypid,2384.0,Electronic,mbrainz_3,Statra Recordings,mbrainz_3,The Sign's Alive,Sign's,United States,2000-09-05,"[The Sign's Alive (original mix), The Sign's A...",0.700000,"{'_id_rule': 'first_non_null', '_id_inputs': [..."
3,discogs_101993,"[mbrainz_4, discogs_6, discogs_101993, lastFM_...","[musicbrainz, discogs, discogs, lastfm, lastfm]",Petalpusher,1632.0,Electronic,mbrainz_4,Naked Music Recordings,mbrainz_4,Surrender,Surrender,United States,1999-04-27,"[Surrender (Petalpusher original), Surrender (...",0.650000,"{'_id_rule': 'first_non_null', '_id_inputs': [..."
4,discogs_11,"[mbrainz_6, discogs_11]","[musicbrainz, discogs]","Garnier, Laurent",5938.0,Electronic,mbrainz_6,F Communications,mbrainz_6,Unreasonable Behaviour,Unreasonable,France,2000-07-24,"[The Warning, City Sphere, Forgotten Thoughts,...",0.703125,"{'_id_rule': 'first_non_null', '_id_inputs': [..."


In [28]:
from PyDI.evaluation import evaluate_schema_consistency, write_schema_consistency_report

consistency_result = evaluate_schema_consistency(
    fused,
    INPUT_DIR / "schemamatching" / "target_schema.json",
    taxonomy_base_path=NOTEBOOK_DIR,
)
write_schema_consistency_report(
    consistency_result,
    OUTPUT_DIR / "metrics" / "consistency.json",
    metadata={
        "usecase": "music",
        "schema_path": str(INPUT_DIR / "schemamatching" / "target_schema.json"),
        "fusion_output_path": str(OUTPUT_DIR / "data_fusion" / "debug_fusion.jsonl"),
        "rows_evaluated": len(fused),
        "fused_table_source": "music_workflow.fused",
        "identifier_columns_excluded": True,
        "open_taxonomy_policy": "non_exhaustive_membership_is_diagnostic_not_penalizing",
    },
)
consistency_result["consistency_score"]

0.9390179113539769

## Step 3: Evaluate Data Fusion

In [29]:
from PyDI.fusion import tokenized_match, year_only_match, set_equality_match, numeric_tolerance_match, intersection


strategy.add_evaluation_function("name", tokenized_match)
strategy.add_evaluation_function("artist", tokenized_match)
strategy.add_evaluation_function("duration", numeric_tolerance_match, tolerance=10) # 10 seconds tolerance
strategy.add_evaluation_function("release-date", year_only_match)
strategy.add_evaluation_function("release-country", tokenized_match)
strategy.add_evaluation_function("label", tokenized_match)
strategy.add_evaluation_function("tracks", tokenized_match)

In [30]:
from PyDI.fusion import DataFusionEvaluator

fusion_test_set = load_xml(INPUT_DIR / 'fusion' / 'test_set_final.xml', name='fusion_test_set', nested_handling='aggregate')

# normalize country names in test set reusing mapping from discogs dataset
translator = SchemaTranslator()
spec.set_column("release-country", country_format="name")
mapping = discogs_mapping.copy()
mapping["source_dataset"] = "fusion_test_set"
mapping["source_column"] = mapping["target_column"] # columns already have the correct name
fusion_test_set = translator.translate(
    fusion_test_set, mapping,
    normalize=spec, on_failure="keep"
)

if "tracks" in fusion_test_set.columns:
    fusion_test_set["tracks"] = fusion_test_set["tracks"].apply(parse_track_list)

# Create evaluator with our fusion strategy
evaluator = DataFusionEvaluator(strategy, debug=True, debug_file=OUTPUT_DIR / "data_fusion" / "debug_fusion_eval.jsonl", debug_format="json")

# Evaluate the fused results against the gold standard
print("Evaluating fusion results against gold standard...")
evaluation_results = evaluator.evaluate(
    fused_df=fused,
    fused_id_column='id',
    gold_df=fusion_test_set,
    gold_id_column='id',
)

# Display evaluation metrics
print("\nFusion Evaluation Results:")
print("=" * 40)
for metric, value in evaluation_results.items():
    if isinstance(value, float):
        print(f"  {metric}: {value:.3f}")
    else:
        print(f"  {metric}: {value}")
        
print(f"\nOverall Accuracy: {evaluation_results.get('overall_accuracy', 0):.1%}")

Evaluating fusion results against gold standard...


[WARNING] PyDI.fusion.evaluation - Missing 3 expected/reference records in fused dataset: mbrainz_13875, mbrainz_188, mbrainz_5321



Fusion Evaluation Results:
  overall_accuracy: 0.660
  macro_accuracy: 0.652
  num_evaluated_records: 97
  num_evaluated_attributes: 8
  total_evaluations: 746
  total_correct: 492
  release-country_accuracy: 0.825
  release-country_count: 97
  artist_accuracy: 0.711
  artist_count: 97
  genre_accuracy: 0.378
  genre_count: 74
  duration_accuracy: 0.557
  duration_count: 97
  name_accuracy: 0.732
  name_count: 97
  release-date_accuracy: 0.753
  release-date_count: 97
  label_accuracy: 0.722
  label_count: 90
  tracks_accuracy: 0.536
  tracks_count: 97

Overall Accuracy: 66.0%


In [31]:
end_time = time.time()
elapsed_time = end_time - start_time
print(f"\nTotal execution time: {elapsed_time:.2f} seconds")


Total execution time: 184.28 seconds
